<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_14_4_deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 14: Wrapping Up**  

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 14 Material

* Part 14.1: Model Drift [[Notebook]](t81_558_class_14_1_drift.ipynb)
* Part 14.2: Dealing with Bias [[Notebook]](t81_558_class_14_2_bias.ipynb)
* Part 14.3: Other Deep Learning Frameworks [[Notebook]](t81_558_class_14_3_frameworks.ipynb)
* **Part 14.4: Deploying a PyTorch Neural Network** [[Notebook]](t81_558_class_14_4_deploy.ipynb)
* Part 14.5: The Future of AI [[Notebook]](t81_558_class_14_5_new_tech.ipynb)

# Google CoLab Instructions

The following code checks that Google CoLab is running and sets up the correct hardware settings for PyTorch.

In [1]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 2.5)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: not using Google CoLab


Using device: mps


# Part 14.4: Deploying a PyTorch Neural Network

Training a model is only half the work; a model creates value only once it is *deployed*, making predictions on real inputs in a real application. Deployment introduces a new set of concerns that have nothing to do with accuracy: how to save the model reliably, how to run it without the original training code, how to serve predictions quickly to many users, and how to keep inference efficient. This section covers the practical path from a trained PyTorch model to a production-ready one.

The good news is that PyTorch provides mature tools for every step, most of them built into the core library. We will save and reload a model the recommended way, convert it into a self-contained serialized form with TorchScript that runs without Python source code, and write a minimal prediction function of the kind that would sit behind a web endpoint. Along the way we note the wider ecosystem, from ONNX to specialized serving systems, that production teams rely on.

## Saving and Loading a Model

The first requirement of deployment is persistence: saving a trained model to disk and loading it back exactly. PyTorch's recommended approach is to save the **state dictionary**, the tensor of learned parameters, rather than the whole Python object. Saving the state dict is robust because it does not depend on the exact file layout of your code, whereas saving the entire model pickles the class definition and can break when your code is refactored.

Loading is a two-step process: recreate the model architecture, then load the saved parameters into it. Two details matter for inference. Calling `model.eval()` switches layers like dropout and batch normalization into evaluation mode, and wrapping predictions in `torch.no_grad()` disables gradient tracking, saving memory and time. We first train a small model to have something to deploy.

In [2]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.datasets import load_iris

X_np, y_np = load_iris(return_X_y=True)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)


class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 3)
        self.act = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


model = IrisNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
for _ in range(300):
    opt.zero_grad()
    loss_fn(model(X), y).backward()
    opt.step()

# Save ONLY the learned parameters (the recommended approach).
torch.save(model.state_dict(), "iris_model.pt")
print("saved iris_model.pt")

# Load: recreate the architecture, then load the parameters into it.
loaded = IrisNet()
loaded.load_state_dict(torch.load("iris_model.pt"))
loaded.eval()                       # switch to inference mode

with torch.no_grad():
    same = torch.allclose(model(X), loaded(X))
print("reloaded model reproduces the original outputs:", same)

saved iris_model.pt
reloaded model reproduces the original outputs: True


## Serialization with TorchScript

Loading a state dict still requires the original Python class definition to be available. In production you often want the model to be fully self-contained, runnable in an environment that does not have your training code, or even without Python at all. **TorchScript** serializes both the model's parameters and its computation into a single portable artifact that a TorchScript runtime can execute in Python or C++.

There are two ways to produce a TorchScript module. `torch.jit.trace` runs an example input through the model and records the operations it performs, which works well for straightforward feed-forward networks. `torch.jit.script` compiles the model's code directly, correctly handling data-dependent control flow such as loops and conditionals. We trace our model, save it, and confirm the reloaded TorchScript module, which carries no dependency on the `IrisNet` class, produces the same predictions.

In [3]:
# Trace the model into a self-contained TorchScript artifact.
example = torch.randn(1, 4)
scripted = torch.jit.trace(loaded, example)
scripted.save("iris_scripted.pt")
print("saved iris_scripted.pt (self-contained; no Python class needed)")

# Reload WITHOUT any reference to the IrisNet class definition.
runtime_model = torch.jit.load("iris_scripted.pt")
runtime_model.eval()

with torch.no_grad():
    match = torch.allclose(loaded(X), runtime_model(X), atol=1e-6)
print("TorchScript model matches the original:", match)

saved iris_scripted.pt (self-contained; no Python class needed)
TorchScript model matches the original: True


## Serving Predictions

A deployed model usually sits behind a small function that accepts raw input, runs the model, and returns a human-meaningful result. In a real service this function would be wrapped in a web framework such as FastAPI or Flask and exposed as a REST endpoint, or hosted by a dedicated system like TorchServe that handles batching, versioning, and scaling. The core prediction logic, however, is exactly what you see below: put the model in evaluation mode, disable gradients, run the forward pass, and translate the output tensor into a label and a confidence.

In [4]:
iris_classes = ["setosa", "versicolor", "virginica"]


def predict(features):
    """Take a raw list of 4 measurements and return a label and confidence."""
    x = torch.tensor([features], dtype=torch.float32)
    with torch.no_grad():
        probs = torch.softmax(runtime_model(x), dim=1)[0]
    idx = int(probs.argmax())
    return {"class": iris_classes[idx], "confidence": float(probs[idx])}


# Simulate a request hitting the endpoint.
sample = [5.1, 3.5, 1.4, 0.2]
result = predict(sample)
print(f"input measurements: {sample}")
print(f"prediction: {result['class']}  (confidence {result['confidence']:.1%})")

input measurements: [5.1, 3.5, 1.4, 0.2]
prediction: setosa  (confidence 99.9%)


The prediction function turns the model into a usable service: raw numbers in, a labeled answer with a confidence out. Everything else in a production deployment, the web framework, load balancing, autoscaling, monitoring, is infrastructure built around this small core.

Beyond what we have shown, several tools address specific deployment needs. **ONNX** exports a model to a framework-neutral format that runs on optimized runtimes across many platforms. **Quantization** and **half-precision** shrink models and speed up inference by using lower-precision numbers. **ExecuTorch** targets phones and edge devices, and **TensorRT** squeezes maximum performance from NVIDIA GPUs. And as [Part 14.1](t81_558_class_14_1_drift.ipynb) stressed, deployment is not the end: a served model must be monitored for drift and retrained as the world changes. Having covered the full life cycle from training to production, the course closes by looking ahead, to [The Future of AI](t81_558_class_14_5_new_tech.ipynb).